# Lab 3.2 — NLP APIs: Azure Text Analytics, Sentiment & Opinion Mining
### Module 4: Large Language Models | Chitkara University B-Tech AI Specialization
---
**Duration:** 75 minutes | **Mode:** Individual | **Day 3 — Wednesday**

> **Scenario:** You have joined the product analytics team at Flipkart-style
> e-commerce platform called KartBazaar. The team wants to automatically mine
> product review data to extract: (1) overall sentiment, (2) named entities
> like brand names and products, and (3) aspect-based sentiment — not just
> 'this review is negative' but 'the *battery life* is BAD while the *camera* is EXCELLENT'.
> You have 20 product reviews to process and must produce a dashboard summary.

**Objective:** Use a cloud NLP API to perform sentiment analysis, named entity
recognition (NER), and opinion mining (aspect-based sentiment) on product reviews.
Visualise results in charts and compare API predictions to your own human judgement.

---
## Choose your NLP backend — read this carefully before running

This lab supports **three backends**:

| Option | Service | Cost | Setup |
|--------|---------|------|-------|
| **A — Azure AI Language** | Azure Cognitive Services | Free F0: 5,000 calls/month | ~10 min |
| **B — HuggingFace Models** | Free open-source models | Completely free | ~2 min |
| **C — TextBlob + NLTK** | Local Python libraries | Completely free | 0 min |

**Recommendation:** Option A gives the most realistic enterprise experience and
supports opinion mining (aspect-level sentiment) which Options B/C cannot do as well.
Option B is a solid free alternative. Option C is the simplest fallback.

---
## OPTION A — Azure AI Language Setup (step-by-step)

### Step 1 — Create Azure AI Language resource
1. Log into [portal.azure.com](https://portal.azure.com)
   (Use the same account from Lab 2.1 — your free credits are still valid)
2. Click **+ Create a resource** → search **Language** → select **Language service**
3. Click **Create** and fill in:
   - **Subscription:** Your free trial subscription
   - **Resource group:** `chitkara-lab` (same as before)
   - **Region:** `East US` (most features available here)
   - **Name:** `chitkara-language-YOUR_NAME` (must be unique)
   - **Pricing tier:** `Free F0` — 5,000 transactions/month at no cost
4. Leave all feature selections as default → **Review + Create → Create**
5. Wait ~2 minutes for deployment

### Step 2 — Get your credentials
1. Go to your new Language resource
2. Left sidebar → **Keys and Endpoint**
3. Copy **Key 1** and the **Endpoint URL** (format: `https://YOUR-NAME.cognitiveservices.azure.com/`)
4. Paste them in the cell below

### Step 3 — Note what the Free F0 tier includes
- 5,000 text records / month for sentiment analysis
- 5,000 text records / month for NER
- 5,000 text records / month for key phrase extraction
- 5,000 text records / month for opinion mining
- This lab uses ~80 API calls total — well within the free tier

---
## OPTION B — HuggingFace (free, no account needed for inference)

We use pre-trained models from the HuggingFace Hub directly.
Models run on Colab's CPU — no API key needed, but slower than Azure.
Set `USE_HF_LOCAL = True` and run.

---
## OPTION C — TextBlob (zero setup)

TextBlob is a lightweight Python NLP library. Much simpler than Azure/BERT
but still demonstrates the concepts. Set `USE_TEXTBLOB = True`.


## Task 1 — Configure your backend and create the review dataset

We create 20 realistic product reviews covering smartphones, headphones, and laptops.
Each review has a known sentiment polarity for ground-truth comparison.


In [ ]:
# ── CHOOSE YOUR BACKEND — set exactly ONE to True ────────────────────────
USE_AZURE_LANG  = False    # Azure AI Language (best features)
AZURE_LANG_KEY  = 'YOUR_KEY_HERE'
AZURE_LANG_ENDPOINT = 'https://YOUR-NAME.cognitiveservices.azure.com/'

USE_HF_LOCAL    = False    # HuggingFace models running locally on Colab

USE_TEXTBLOB    = True     # TextBlob — zero setup, works immediately

active = [n for n,f in [('Azure AI Language',USE_AZURE_LANG),
                         ('HuggingFace Local',USE_HF_LOCAL),
                         ('TextBlob',USE_TEXTBLOB)] if f]
print(f'NLP backend: {active[0] if active else "NONE — set one to True"}')


In [ ]:
!pip install azure-ai-textanalytics textblob transformers torch --quiet

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import re, json, warnings, time
warnings.filterwarnings('ignore')

print('Packages installed')


In [ ]:
# ── 20 product reviews with ground-truth labels ──────────────────────────
# Format: (review_text, product_category, true_sentiment)
# true_sentiment: 'positive', 'negative', 'mixed', 'neutral'
reviews = [
    # SMARTPHONES (7)
    ('This phone is absolutely incredible. The camera captures stunning detail '
     'even in low light and the battery easily lasts two full days. Best purchase '
     'I have made in years.', 'smartphone', 'positive'),

    ('Battery life is disappointing — drains completely in 6 hours of normal use. '
     'The camera is okay but nothing special for this price range. '
     'Expected much better from this brand.', 'smartphone', 'negative'),

    ('The display is gorgeous with vibrant colours and silky smooth scrolling. '
     'However the phone gets uncomfortably warm during gaming sessions '
     'and the charger in the box is only 18W — feels cheap for a flagship device.', 'smartphone', 'mixed'),

    ('Decent phone for the price. Does everything I need — calls, WhatsApp, some gaming. '
     'Nothing exceptional but nothing terrible either.', 'smartphone', 'neutral'),

    ('Brilliant! Upgraded from a 4-year-old phone and the difference is night and day. '
     'The face unlock is instant, the stereo speakers are loud and clear, '
     'and the build quality feels premium.', 'smartphone', 'positive'),

    ('Terrible camera in daylight — photos look washed out and overexposed every time. '
     'Tried updating the software but no improvement. Very frustrated given the price paid.', 'smartphone', 'negative'),

    ('Good performance, smooth UI, reasonable camera. The weight is slightly heavy '
     'but you get used to it. Overall satisfied with this purchase.', 'smartphone', 'positive'),

    # HEADPHONES (7)
    ('The noise cancellation on these headphones is phenomenal. '
     'Even in a crowded office I cannot hear anything around me. '
     'The sound quality is rich and detailed. Completely worth the price.', 'headphone', 'positive'),

    ('Very uncomfortable after 30 minutes. The ear cups press hard against my ears '
     'and the headband is tight even on the smallest adjustment. '
     'Returned after one day.', 'headphone', 'negative'),

    ('Sound quality is exceptional — deep bass, clear mids, crisp highs. '
     'The ANC could be stronger though, it only partially blocks wind noise outdoors. '
     'Battery life of 30 hours is excellent.', 'headphone', 'mixed'),

    ('Lightweight and comfortable. Good for long calls at work. '
     'Microphone picks up voice clearly. Nothing audiophile-grade but great for the price.', 'headphone', 'positive'),

    ('The Bluetooth connection drops every 10 minutes when my phone is in my pocket. '
     'This is unusable for commuting. Very disappointed with such a basic failure.', 'headphone', 'negative'),

    ('Average in every dimension. Sound is fine, comfort is okay, '
     'call quality acceptable. Not exceptional but not bad. A safe mediocre choice.', 'headphone', 'neutral'),

    ('Great sound and long battery life but the touch controls are way too sensitive. '
     'Kept pausing music accidentally when adjusting the fit. '
     'Firmware update hopefully fixes this.', 'headphone', 'mixed'),

    # LAPTOPS (6)
    ('Incredibly fast with the M3 chip — everything is instant. '
     'The keyboard is a pleasure to type on and the display makes every colour pop. '
     'Worth every rupee if you are a professional.', 'laptop', 'positive'),

    ('Fan noise is ridiculous under any load. The keyboard flexes when typing. '
     'Trackpad is laggy and imprecise. For this price this is simply not acceptable.', 'laptop', 'negative'),

    ('Performance is excellent for development work. Build quality is solid. '
     'The only drawback is only two USB-C ports — I need a hub for everything.', 'laptop', 'mixed'),

    ('Standard business laptop. Reliable, long battery life, reasonable display. '
     'No complaints and no outstanding features. Gets the job done.', 'laptop', 'neutral'),

    ('Lightning fast SSD, gorgeous display, runs silent even under load. '
     'Best laptop I have owned in 10 years. No hesitation recommending this.', 'laptop', 'positive'),

    ('Overheats within 20 minutes of video editing. Throttles to half speed to cool down. '
     'The whole point of buying a powerful laptop is performance — this fails at its one job.', 'laptop', 'negative'),
]

print(f'Review dataset: {len(reviews)} reviews')
cats = pd.Series([r[1] for r in reviews]).value_counts()
sents= pd.Series([r[2] for r in reviews]).value_counts()
print('By product category:', dict(cats))
print('By true sentiment:  ', dict(sents))


## Task 2 — Sentiment Analysis

**Document-level sentiment analysis** classifies an entire piece of text
as positive, negative, neutral, or mixed, along with a confidence score.

**Why confidence scores matter:**
A review with positive=0.95 is very clearly positive.
A review with positive=0.52 is borderline — you might want human review.
Routing borderline cases to humans is a key production pattern.

We run all 20 reviews through sentiment analysis and then compare the API
predictions to our ground-truth labels.


In [ ]:
def run_sentiment_analysis(texts):
    # Run sentiment analysis using the configured backend.
    # Returns list of {'sentiment': str, 'positive': float, 'negative': float, 'neutral': float}
    results = []

    if USE_AZURE_LANG:
        from azure.ai.textanalytics import TextAnalyticsClient
        from azure.core.credentials import AzureKeyCredential
        client = TextAnalyticsClient(
            endpoint=AZURE_LANG_ENDPOINT,
            credential=AzureKeyCredential(AZURE_LANG_KEY)
        )
        # Process in batches of 10 (Azure limit per call)
        for i in range(0, len(texts), 10):
            batch = texts[i:i+10]
            docs  = [{'id': str(j), 'text': t, 'language': 'en'} for j, t in enumerate(batch)]
            resp  = client.analyze_sentiment(docs)
            for doc in resp:
                if not doc.is_error:
                    s = doc.confidence_scores
                    results.append({'sentiment': doc.sentiment,
                                    'positive': s.positive,
                                    'negative': s.negative,
                                    'neutral':  s.neutral})
                else:
                    results.append({'sentiment': 'error', 'positive':0,'negative':0,'neutral':0})

    elif USE_HF_LOCAL:
        from transformers import pipeline as hp
        print('Loading DistilBERT sentiment model (~250MB)...')
        pipe = hp('sentiment-analysis',
                  model='distilbert-base-uncased-finetuned-sst-2-english',
                  device=-1, truncation=True, max_length=512)
        for text in texts:
            out = pipe(text[:512])[0]
            label = out['label'].lower()
            score = out['score']
            results.append({
                'sentiment': 'positive' if label=='positive' else 'negative',
                'positive':  score if label=='positive' else 1-score,
                'negative':  score if label=='negative' else 1-score,
                'neutral':   0.0
            })

    else:  # TEXTBLOB
        from textblob import TextBlob
        for text in texts:
            blob       = TextBlob(text)
            polarity   = blob.sentiment.polarity  # -1 to +1
            if polarity > 0.15:
                sent, pos, neg, neu = 'positive', min(polarity+0.5, 1.0), 0.05, 0.1
            elif polarity < -0.15:
                sent, pos, neg, neu = 'negative', 0.05, min(-polarity+0.5, 1.0), 0.1
            else:
                sent, pos, neg, neu = 'neutral',  0.2,  0.2,  0.6
            results.append({'sentiment': sent, 'positive': pos, 'negative': neg, 'neutral': neu})

    return results

texts_only = [r[0] for r in reviews]
print('Running sentiment analysis on 20 reviews...')
sentiment_results = run_sentiment_analysis(texts_only)

print(f'Done! Results for first 5 reviews:')
for i, (res, (text, cat, true_sent)) in enumerate(zip(sentiment_results[:5], reviews[:5])):
    print(f'\n  [{i+1}] Product: {cat}')
    print(f'  Text: {text[:80]}...')
    print(f'  True sentiment   : {true_sent}')
    print(f'  Predicted        : {res["sentiment"]}')
    print(f'  Confidence scores: pos={res["positive"]:.2f}  neg={res["negative"]:.2f}  neu={res["neutral"]:.2f}')


In [ ]:
# ── Sentiment accuracy and visualisation ─────────────────────────────────
true_labels = [r[2] for r in reviews]

# Map 'mixed' true labels: if predicted is positive or negative, count as correct
# (Most models don't output 'mixed' — they pick the dominant signal)
def is_correct(pred, true):
    if true == 'mixed':
        return pred in ('positive', 'negative', 'mixed')
    return pred == true

correct = sum(is_correct(res['sentiment'], true)
              for res, true in zip(sentiment_results, true_labels))
accuracy = correct / len(reviews)

print(f'SENTIMENT ANALYSIS ACCURACY: {correct}/{len(reviews)} = {accuracy:.0%}')
print()

# Build comparison dataframe
df_sent = pd.DataFrame([
    {
        'Review': text[:55] + '...',
        'Category'  : cat,
        'True'      : true,
        'Predicted' : res['sentiment'],
        'Pos Score' : round(res['positive'], 3),
        'Neg Score' : round(res['negative'], 3),
        'Correct'   : is_correct(res['sentiment'], true),
    }
    for res, (text, cat, true) in zip(sentiment_results, reviews)
])

print('FULL RESULTS TABLE:')
print(df_sent[['Category','True','Predicted','Pos Score','Neg Score','Correct']].to_string(index=False))


In [ ]:
# ── Sentiment visualisations ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Chart 1: Sentiment distribution (predicted)
pred_sent_counts = pd.Series([r['sentiment'] for r in sentiment_results]).value_counts()
sent_colors = {'positive':'#059669','negative':'#DC2626','neutral':'#6B7280','mixed':'#F59E0B'}
axes[0].bar(pred_sent_counts.index,
            pred_sent_counts.values,
            color=[sent_colors.get(s,'#888') for s in pred_sent_counts.index],
            edgecolor='white', alpha=0.9)
for i, (idx, val) in enumerate(pred_sent_counts.items()):
    axes[0].text(i, val+0.1, str(val), ha='center', fontweight='bold', fontsize=12)
axes[0].set_title('Predicted Sentiment Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].spines[['top','right']].set_visible(False)

# Chart 2: Average confidence score per true sentiment category
for sent_type in ['positive','negative']:
    idxs = [i for i,(_, _, t) in enumerate(reviews) if t==sent_type]
    pos_scores = [sentiment_results[i]['positive'] for i in idxs]
    neg_scores = [sentiment_results[i]['negative'] for i in idxs]
    # Not used directly but informative to compute:

all_pos = [r['positive'] for r in sentiment_results]
all_neg = [r['negative'] for r in sentiment_results]
axes[1].scatter(all_pos, all_neg,
                c=[sent_colors.get(r['sentiment'],'#888') for r in sentiment_results],
                s=80, alpha=0.85, edgecolors='white', linewidth=0.8)
axes[1].set_xlabel('Positive Score')
axes[1].set_ylabel('Negative Score')
axes[1].set_title('Positive vs Negative Score per Review', fontweight='bold')
axes[1].spines[['top','right']].set_visible(False)
axes[1].axline((0.5, 0.5), slope=-1, color='grey', linestyle='--', alpha=0.4, label='50/50 line')

# Chart 3: Accuracy per product category
for i, cat in enumerate(['smartphone','headphone','laptop']):
    cat_idxs = [j for j,(_, c, _) in enumerate(reviews) if c==cat]
    cat_acc  = sum(1 for j in cat_idxs if df_sent.iloc[j]['Correct']) / len(cat_idxs)
    axes[2].bar(i, cat_acc*100,
                color=['#7C3AED','#0891B2','#F59E0B'][i],
                edgecolor='white', alpha=0.9, width=0.5)
    axes[2].text(i, cat_acc*100+1, f'{cat_acc:.0%}', ha='center', fontweight='bold', fontsize=12)
axes[2].set_xticks(range(3))
axes[2].set_xticklabels(['Smartphone','Headphone','Laptop'])
axes[2].set_ylabel('Accuracy (%)')
axes[2].set_ylim(0,115)
axes[2].set_title('Sentiment Accuracy by Product Category', fontweight='bold')
axes[2].spines[['top','right']].set_visible(False)

plt.suptitle('Sentiment Analysis Dashboard — 20 Product Reviews', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('sentiment_dashboard.png', dpi=130, bbox_inches='tight')
plt.show()
print('Dashboard saved as sentiment_dashboard.png')


## Task 3 — Named Entity Recognition (NER)

**Named Entity Recognition** identifies and classifies specific named things
in text: product names, brands, locations, quantities, dates, and more.

For our product reviews, NER can automatically extract:
- **Product names:** 'Sony WH-1000XM5', 'M3 chip', 'WhatsApp'
- **Quantities:** '30 hours', '18W', '2 full days'
- **Organizations:** brand names mentioned in reviews

This is valuable for:
- Automatically tagging which product a review refers to
- Tracking specific features mentioned (battery = X hours)
- Competitive analysis (which competitor brands appear in your reviews?)


In [ ]:
def run_ner(texts):
    # Run NER using the configured backend.
    # Returns list of lists: each inner list has {'text', 'category', 'confidence'} dicts.
    results = [[] for _ in texts]

    if USE_AZURE_LANG:
        from azure.ai.textanalytics import TextAnalyticsClient
        from azure.core.credentials import AzureKeyCredential
        client = TextAnalyticsClient(endpoint=AZURE_LANG_ENDPOINT,
                                     credential=AzureKeyCredential(AZURE_LANG_KEY))
        for i in range(0, len(texts), 10):
            batch = [{'id': str(j), 'text': t, 'language': 'en'}
                     for j, t in enumerate(texts[i:i+10])]
            resp  = client.recognize_entities(batch)
            for j, doc in enumerate(resp):
                if not doc.is_error:
                    results[i+j] = [{'text': e.text, 'category': e.category,
                                      'confidence': round(e.confidence_score, 3)}
                                     for e in doc.entities]

    elif USE_HF_LOCAL:
        from transformers import pipeline as hp
        print('Loading NER model (bert-base-NER)...')
        pipe = hp('ner', model='dbmdz/bert-large-cased-finetuned-conll03-english',
                  aggregation_strategy='simple', device=-1)
        for i, text in enumerate(texts):
            ents = pipe(text[:512])
            results[i] = [{'text': e['word'], 'category': e['entity_group'],
                            'confidence': round(e['score'], 3)} for e in ents]

    else:  # TEXTBLOB — basic noun phrase extraction as fallback
        from textblob import TextBlob
        import re
        brand_patterns = r'\b(Sony|Samsung|Apple|OnePlus|Xiaomi|Realme|Oppo|Vivo|'\
                         r'Bose|JBL|Sennheiser|Dell|HP|Lenovo|Asus|Acer|LG|'\
                         r'WhatsApp|YouTube|Netflix|Bluetooth|USB)\b'
        qty_patterns   = r'\b(\d+\s*(?:hours?|W|GB|TB|MHz|Hz|mm|inch|hrs?|days?))\b'
        for i, text in enumerate(texts):
            ents = []
            for m in re.finditer(brand_patterns, text, re.IGNORECASE):
                ents.append({'text': m.group(), 'category': 'Brand/Product', 'confidence': 0.90})
            for m in re.finditer(qty_patterns, text, re.IGNORECASE):
                ents.append({'text': m.group(), 'category': 'Quantity', 'confidence': 0.85})
            results[i] = ents

    return results

print('Running Named Entity Recognition on 20 reviews...')
ner_results = run_ner(texts_only)

print('\nNER results for first 5 reviews:')
for i, (ents, (text, cat, true_sent)) in enumerate(zip(ner_results[:5], reviews[:5])):
    print(f'\n  [{i+1}] {text[:80]}...')
    if ents:
        for e in ents:
            print(f'    [{e["category"]:20}]  "{e["text"]}"  (conf={e["confidence"]})')
    else:
        print('    No entities detected')


In [ ]:
# ── Entity frequency analysis ─────────────────────────────────────────────
from collections import Counter

all_entities    = [(e['text'].lower(), e['category'])
                   for ents in ner_results for e in ents]
entity_counts   = Counter(text for text, _ in all_entities)
category_counts = Counter(cat for _, cat in all_entities)

print(f'Total entities found: {len(all_entities)}')
print(f'\nTop-15 most frequent entities:')
for entity, count in entity_counts.most_common(15):
    print(f'  {count:3d}x  "{entity}"')

print(f'\nEntity category distribution:')
for cat, count in category_counts.most_common():
    bar = chr(9608) * count
    print(f'  {cat:<25}: {count:3d}  {bar}')

# Visualise top entities bar chart
top15 = entity_counts.most_common(12)
if top15:
    ent_labels = [e for e, _ in top15]
    ent_counts = [c for _, c in top15]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(ent_labels[::-1], ent_counts[::-1], color='#7C3AED', alpha=0.85, edgecolor='white')
    ax.set_xlabel('Frequency across 20 reviews')
    ax.set_title('Top Named Entities Detected in Product Reviews', fontweight='bold')
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('ner_frequency.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('Chart saved as ner_frequency.png')


## Task 4 — Opinion Mining (Aspect-Based Sentiment)

Standard sentiment analysis tells you 'this review is mixed'.
**Opinion mining** (also called aspect-based sentiment analysis, ABSA) goes deeper:
it extracts *which specific aspect* is positive or negative.

**Example:**
- Review: *'The battery life is terrible but the camera is spectacular.'*
- Standard sentiment: `mixed` ✓ (correct but vague)
- Opinion mining:
  - `battery life` → **negative** (target)
  - `camera` → **positive** (target)

This is extremely valuable for product teams:
- Which features are customers consistently praising?
- Which features are driving return requests?
- How does sentiment for 'battery' compare across smartphone brands?

**Note:** Opinion mining is only available in Azure AI Language (Option A).
For Options B/C we use a rule-based fallback that detects aspect words.


In [ ]:
# ── Aspect words we want to track ────────────────────────────────────────
ASPECT_KEYWORDS = {
    'battery'       : ['battery', 'battery life', 'charge', 'drains', 'lasts'],
    'camera'        : ['camera', 'photo', 'picture', 'image', 'video', 'lens', 'capture'],
    'display'       : ['display', 'screen', 'brightness', 'colour', 'colors', 'resolution'],
    'performance'   : ['performance', 'speed', 'fast', 'slow', 'lag', 'smooth', 'powerful'],
    'build_quality' : ['build', 'quality', 'material', 'premium', 'cheap', 'sturdy', 'solid'],
    'comfort'       : ['comfort', 'comfortable', 'uncomfortable', 'ergonomic', 'wear', 'fit'],
    'sound'         : ['sound', 'audio', 'bass', 'treble', 'music', 'speaker', 'microphone'],
    'noise_cancel'  : ['noise cancellation', 'anc', 'noise cancel', 'noise blocking'],
    'price_value'   : ['price', 'value', 'worth', 'expensive', 'cheap', 'cost', 'rupee'],
}

def run_opinion_mining_azure(texts):
    from azure.ai.textanalytics import TextAnalyticsClient
    from azure.core.credentials import AzureKeyCredential
    client = TextAnalyticsClient(endpoint=AZURE_LANG_ENDPOINT,
                                 credential=AzureKeyCredential(AZURE_LANG_KEY))
    all_opinions = [[] for _ in texts]
    for i in range(0, len(texts), 10):
        batch = [{'id': str(j), 'text': t, 'language': 'en'}
                 for j, t in enumerate(texts[i:i+10])]
        resp  = client.analyze_sentiment(batch, show_opinion_mining=True)
        for j, doc in enumerate(resp):
            if not doc.is_error:
                for sentence in doc.sentences:
                    for assessment in sentence.mined_opinions:
                        target  = assessment.target
                        for opinion in assessment.assessments:
                            all_opinions[i+j].append({
                                'target'    : target.text,
                                'sentiment' : opinion.sentiment,
                                'assessment': opinion.text,
                                'confidence': round(opinion.confidence_scores.positive
                                                    if opinion.sentiment=='positive'
                                                    else opinion.confidence_scores.negative, 3)
                            })
    return all_opinions

def run_opinion_mining_rules(texts):
    # Rule-based fallback: find aspect keywords + surrounding sentiment words.
    POSITIVE_WORDS = {'excellent','great','amazing','good','love','fantastic','brilliant',
                      'superb','perfect','wonderful','best','stunning','crisp','rich',
                      'phenomenal','exceptional','gorgeous','instant','silky','loud','clear',
                      'incredible','fast','smooth','premium','solid','long'}
    NEGATIVE_WORDS = {'terrible','bad','poor','awful','disappointing','horrible','worst',
                      'laggy','drops','noisy','uncomfortable','tight','overheats','hot',
                      'slow','cheap','weak','drains','thick','fails','wrong','washed'}

    all_opinions = []
    for text in texts:
        text_lower = text.lower()
        doc_opinions = []
        for aspect, keywords in ASPECT_KEYWORDS.items():
            for kw in keywords:
                pos = text_lower.find(kw)
                if pos != -1:
                    # Check a window of 40 chars around the keyword
                    window = text_lower[max(0, pos-40):pos+40]
                    pos_hits = sum(1 for w in POSITIVE_WORDS if w in window)
                    neg_hits = sum(1 for w in NEGATIVE_WORDS if w in window)
                    if pos_hits > neg_hits:
                        sent, conf = 'positive', 0.75
                    elif neg_hits > pos_hits:
                        sent, conf = 'negative', 0.75
                    else:
                        sent, conf = 'neutral', 0.5
                    doc_opinions.append({'target': aspect.replace('_',' '),
                                         'sentiment': sent, 'assessment': kw,
                                         'confidence': conf})
                    break  # only count each aspect once per review
        all_opinions.append(doc_opinions)
    return all_opinions

print('Running opinion mining on 20 reviews...')
if USE_AZURE_LANG:
    opinion_results = run_opinion_mining_azure(texts_only)
    print('Azure opinion mining complete')
else:
    opinion_results = run_opinion_mining_rules(texts_only)
    print('Rule-based aspect extraction complete')

print('\nOpinion mining results for first 4 reviews:')
for i, (opinions, (text, cat, true_sent)) in enumerate(zip(opinion_results[:4], reviews[:4])):
    print(f'\n  [{i+1}] {text[:90]}...')
    if opinions:
        for op in opinions:
            emoji = '+' if op['sentiment']=='positive' else ('-' if op['sentiment']=='negative' else '~')
            print(f'    [{emoji}]  Target: "{op["target"]}"  |  Assessment: "{op["assessment"]}"  |  Sent: {op["sentiment"]}')
    else:
        print('    No aspect opinions detected')


In [ ]:
# ── Aspect sentiment heatmap ──────────────────────────────────────────────
# Build a matrix: rows = aspects, columns = product categories
ASPECTS = list(ASPECT_KEYWORDS.keys())
CATS    = ['smartphone', 'headphone', 'laptop']

# aspect -> category -> [sentiment_scores] where pos=+1, neg=-1, neu=0
aspect_cat_scores = {a: {c: [] for c in CATS} for a in ASPECTS}

for opinions, (_, cat, _) in zip(opinion_results, reviews):
    for op in opinions:
        target = op['target'].replace(' ', '_').lower()
        # Map target to closest aspect
        matched_aspect = None
        for aspect, keywords in ASPECT_KEYWORDS.items():
            if target in aspect or any(kw in target for kw in keywords[:2]):
                matched_aspect = aspect
                break
        if matched_aspect and cat in CATS:
            score_val = 1 if op['sentiment']=='positive' else (-1 if op['sentiment']=='negative' else 0)
            aspect_cat_scores[matched_aspect][cat].append(score_val)

# Average score per cell
heatmap_data = []
for aspect in ASPECTS:
    row = []
    for cat in CATS:
        scores = aspect_cat_scores[aspect][cat]
        row.append(np.mean(scores) if scores else np.nan)
    heatmap_data.append(row)

heatmap_array = np.array(heatmap_data, dtype=float)

# Plot
fig, ax = plt.subplots(figsize=(9, 8))
import seaborn as sns

# Replace nan with 0 for display
display_array = np.where(np.isnan(heatmap_array), 0, heatmap_array)
sns.heatmap(display_array,
            xticklabels=CATS,
            yticklabels=[a.replace('_',' ') for a in ASPECTS],
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size':10},
            linewidths=0.5, ax=ax)
ax.set_title('Aspect Sentiment Heatmap\n'
             '(+1 = positive, 0 = neutral, -1 = negative)\n'
             'Rows = product aspects  |  Columns = product category',
             fontweight='bold', fontsize=11)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=11)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)
plt.tight_layout()
plt.savefig('aspect_sentiment_heatmap.png', dpi=130, bbox_inches='tight')
plt.show()
print('Heatmap saved as aspect_sentiment_heatmap.png')
print()
print('HOW TO READ: Green = customers are positive about this aspect for this category.')
print('Red = customers are negative. White/blank = aspect not mentioned for this category.')


## Task 5 — Manual comparison: API vs human judgement

The most important calibration exercise is to **read the API output and disagree with it** —
or agree. This builds your intuition for where cloud NLP APIs are reliable vs where
they fail silently.

For 5 carefully chosen reviews, we will:
1. Record the API's sentiment prediction and confidence
2. Record your own human judgement
3. Identify and explain any disagreements
4. Determine whether the API failure is a problem for the use case


In [ ]:
# ── 5 tricky reviews for human vs API comparison ─────────────────────────
# Pick: 2 mixed, 1 sarcastic, 1 qualified positive, 1 highly negative
tricky_indices = [
    2,   # smartphone mixed: 'display is gorgeous... but gets warm'
    9,   # headphone mixed: 'sound quality exceptional... ANC could be stronger'
    13,  # headphone mixed: 'Great sound... touch controls too sensitive'
    10,  # headphone negative: 'Very uncomfortable after 30 minutes'
    17,  # laptop neutral: 'Standard business laptop. Reliable...'
]

print('HUMAN vs API COMPARISON — 5 Tricky Reviews')
print('=' * 72)

for idx in tricky_indices:
    text, cat, true_sent = reviews[idx]
    api_result = sentiment_results[idx]
    api_sent   = api_result['sentiment']

    agree = 'AGREE' if is_correct(api_sent, true_sent) else 'DISAGREE'

    print(f'\nReview [{idx+1}] — Product: {cat}')
    print(f'  Text: {text}')
    print(f'  True label  : {true_sent}')
    print(f'  API predicted: {api_sent}  (pos={api_result["positive"]:.2f}, neg={api_result["negative"]:.2f})')
    print(f'  Verdict     : {agree}')
    print(f'  Your analysis: [FILL IN — why did the API get this right or wrong?]')
    print()

print('Fill in your analysis for each review above.')
print('Consider: Is the disagreement a real failure or just a label definition issue?')


In [ ]:
# ── Final integrated dashboard ────────────────────────────────────────────
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.4)

# Panel 1: Sentiment distribution by category
ax1 = fig.add_subplot(gs[0, 0])
cat_sent = {cat: {'positive':0,'negative':0,'neutral':0,'mixed':0} for cat in CATS}
for res, (_, cat, _) in zip(sentiment_results, reviews):
    s = res['sentiment']
    if s not in cat_sent[cat]:
        cat_sent[cat][s] = 0
    cat_sent[cat][s] += 1

x = np.arange(len(CATS))
w = 0.22
for i, (sent, color) in enumerate([('positive','#059669'),('negative','#DC2626'),
                                     ('neutral','#6B7280'),('mixed','#F59E0B')]):
    vals = [cat_sent[c].get(sent, 0) for c in CATS]
    ax1.bar(x + i*w, vals, w, label=sent, color=color, alpha=0.85, edgecolor='white')
ax1.set_xticks(x + 1.5*w)
ax1.set_xticklabels(CATS, fontsize=9)
ax1.set_title('Sentiment by Category', fontweight='bold', fontsize=10)
ax1.legend(fontsize=7)
ax1.spines[['top','right']].set_visible(False)

# Panel 2: Confidence score distribution
ax2 = fig.add_subplot(gs[0, 1])
pos_scores = [max(r['positive'], r['negative']) for r in sentiment_results]
ax2.hist(pos_scores, bins=10, color='#7C3AED', alpha=0.85, edgecolor='white')
ax2.axvline(0.8, color='#DC2626', linestyle='--', label='0.8 threshold')
ax2.set_xlabel('Max confidence score')
ax2.set_ylabel('Count')
ax2.set_title('Confidence Score Distribution', fontweight='bold', fontsize=10)
ax2.legend(fontsize=8)
ax2.spines[['top','right']].set_visible(False)

# Panel 3: API accuracy by confidence band
ax3 = fig.add_subplot(gs[0, 2])
bands = [(0.5, 0.7, 'Low (0.5-0.7)'), (0.7, 0.85, 'Mid (0.7-0.85)'), (0.85, 1.01, 'High (>0.85)')]
band_acc = []
band_labels = []
for lo, hi, label in bands:
    idxs = [i for i, r in enumerate(sentiment_results)
            if lo <= max(r['positive'], r['negative'], r['neutral']) < hi]
    if idxs:
        acc = sum(1 for i in idxs if is_correct(sentiment_results[i]['sentiment'], reviews[i][2]))/len(idxs)
        band_acc.append(acc*100)
        band_labels.append(f'{label}\n(n={len(idxs)})')
ax3.bar(range(len(band_acc)), band_acc, color=['#F59E0B','#0891B2','#059669'],
        edgecolor='white', alpha=0.9)
for i, acc in enumerate(band_acc):
    ax3.text(i, acc+1, f'{acc:.0f}%', ha='center', fontweight='bold')
ax3.set_xticks(range(len(band_labels)))
ax3.set_xticklabels(band_labels, fontsize=8)
ax3.set_ylabel('Accuracy (%)')
ax3.set_title('Accuracy by Confidence Band\n(High confidence = more reliable?)', fontweight='bold', fontsize=9)
ax3.set_ylim(0, 115)
ax3.spines[['top','right']].set_visible(False)

# Panel 4 (full-width bottom): Entity category breakdown
ax4 = fig.add_subplot(gs[1, :])
if category_counts:
    cats_ner = list(category_counts.keys())[:10]
    cnts_ner = [category_counts[c] for c in cats_ner]
    ax4.bar(cats_ner, cnts_ner, color='#0891B2', alpha=0.85, edgecolor='white')
    for i, cnt in enumerate(cnts_ner):
        ax4.text(i, cnt+0.1, str(cnt), ha='center', fontweight='bold')
    ax4.set_ylabel('Count')
    ax4.set_title('Named Entity Categories Detected Across All 20 Reviews', fontweight='bold', fontsize=10)
    ax4.set_xticklabels(cats_ner, rotation=20, ha='right')
    ax4.spines[['top','right']].set_visible(False)

plt.suptitle('KartBazaar Product Review NLP Dashboard — 20 Reviews',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig('nlp_dashboard_complete.png', dpi=130, bbox_inches='tight')
plt.show()
print('Complete dashboard saved as nlp_dashboard_complete.png')


## Lab Complete Checklist

- [ ] Backend configured (Azure / HuggingFace / TextBlob)
- [ ] 20 reviews loaded with categories and ground-truth sentiment labels
- [ ] Sentiment analysis run on all 20 reviews
- [ ] Accuracy computed and per-category breakdown printed
- [ ] Sentiment dashboard saved as `sentiment_dashboard.png`
- [ ] NER run on all 20 reviews
- [ ] Top-15 entity frequency chart saved as `ner_frequency.png`
- [ ] Opinion mining run and aspect sentiment extracted
- [ ] Aspect sentiment heatmap saved as `aspect_sentiment_heatmap.png`
- [ ] Human vs API comparison completed for 5 tricky reviews
- [ ] Final integrated dashboard saved as `nlp_dashboard_complete.png`

---
## Reflection Questions

1. **Confidence calibration:** Look at your 'Accuracy by Confidence Band' chart.
   Is the API more accurate when it is more confident? What are the production
   implications of routing low-confidence predictions to a human reviewer?

2. **Mixed sentiment challenge:** Several reviews in the dataset are genuinely 'mixed'.
   The API returns a single sentiment label. How would you redesign the system
   to handle mixed sentiment reviews more usefully for a product team?

3. **Opinion mining value:** Your aspect-level heatmap shows which product features
   customers are positive/negative about. Give a specific business decision that
   a product manager at KartBazaar could make based on this data alone.

4. **NER for competitive intelligence:** The NER results may have detected competitor
   brand names in reviews (e.g., 'unlike Brand X, this one...').
   Design a pipeline that automatically flags reviews mentioning competitors
   and classifies whether the comparison was favourable or unfavourable.

5. **API vs fine-tuned model:** Azure AI Language is a general-purpose model.
   If you fine-tuned a BERT model specifically on electronics reviews,
   would it perform better or worse than Azure? What data would you need?


In [ ]:
answers = {
    'Q1 - Confidence calibration and human-in-loop routing' : 'YOUR ANSWER HERE',
    'Q2 - How to handle mixed sentiment reviews'            : 'YOUR ANSWER HERE',
    'Q3 - Business decision from aspect sentiment'          : 'YOUR ANSWER HERE',
    'Q4 - Competitor mention pipeline design'               : 'YOUR ANSWER HERE',
    'Q5 - Azure general model vs domain fine-tuned BERT'    : 'YOUR ANSWER HERE',
}
for q, a in answers.items():
    print(f'{q}:\n  {a}\n')
